In [6]:
import os
import yaml
import copy
import time
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
def detrend_linear(da, dim="time"):
    """
    Remove a best-fit linear trend along `dim` for each grid point.
    Uses an index-based time axis (0..N-1) to avoid datetime scaling issues.
    """
    t = xr.DataArray(np.arange(da.sizes[dim]), dims=dim, coords={dim: da[dim]})

    valid = np.isfinite(da)
    t_valid = t.where(valid)
    da_valid = da.where(valid)

    t_mean = t_valid.mean(dim, skipna=True)
    y_mean = da_valid.mean(dim, skipna=True)

    cov = ((t_valid - t_mean) * (da_valid - y_mean)).mean(dim, skipna=True)
    var = ((t_valid - t_mean) ** 2).mean(dim, skipna=True)

    slope = cov / var
    intercept = y_mean - slope * t_mean

    trend = slope * t + intercept
    return da - trend

In [4]:
# fn_ERA5 = f'/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/ERA5_{year}.zarr'
# ds_ERA5 = xr.open_zarr(fn_ERA5)
# fn_CESM = f'/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/SMYLE_{year-1}-11-01_daily_ensemble.zarr'
# ds_CESM = xr.open_zarr(fn_CESM)

### Preparing gridded yearly metrics

In [5]:
dict_loc = {
    'Pituffik': (76.4, -68.575),
    'Fairbanks': (64.75, -147.4),
    'Guam': (13.475, 144.75),
    'Yuma_PG': (33.125, -114.125),
    'Fort_Bragg': (35.05, -79.115),
}
keys = list(dict_loc.keys())

### Fairbanks: TMAX

In [7]:
key = 'Fairbanks'
dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{key}/'
base_dir = '/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/'

vars_keep = ["TREFHT", "TREFHTMX"]
years = range(1958, 2020)

ds_collection = []

for year in years:

    fn_CESM = f"{base_dir}/{key}/SMYLE_{key}_{year}.zarr"

    # Faster Zarr open in many cases:
    ds = xr.open_zarr(fn_CESM, consolidated=True)[vars_keep].chunk({"time": -1})

    # Slice once
    ds = ds.sel(time=slice(f'{year+1}-01-01', f'{year+10}-12-31'))

    # Groupby objects (cheap)
    g_TREFHT = ds["TREFHT"].groupby("time.year")
    g_TREFHTMX = ds["TREFHTMX"].groupby("time.year")

    # Annual maxima (vectorized)
    da_TREFHTMX_max = g_TREFHTMX.max("time", skipna=True).rename("TREFHTMX_max")
    da_TREFHT_max   = g_TREFHT.max("time", skipna=True).rename("TREFHT_max")

    # Rolling maxima over each year (avoid groupby.map(lambda ...))
    da_TREFHT_7d_max = (
        ds["TREFHT"]
        .rolling(time=7, min_periods=7).mean()
        .groupby("time.year").max("time", skipna=True)
        .rename("TREFHT_7d_max")
    )

    da_TREFHT_30d_max = (
        ds["TREFHT"]
        .rolling(time=30, min_periods=30).mean()
        .groupby("time.year").max("time", skipna=True)
        .rename("TREFHT_30d_max")
    )

    # Merge to dataset
    ds_merge = xr.merge([da_TREFHTMX_max, da_TREFHT_max, da_TREFHT_7d_max, da_TREFHT_30d_max])

    # Keep your year coordinate logic (but avoid re-creating arange via constant)
    ds_merge = ds_merge.assign_coords(year=np.arange(10, dtype=int))

    ds_collection.append(ds_merge)

ds_all = xr.concat(ds_collection, dim="init_time")
ds_all = ds_all.assign_coords(init_time=np.arange(1959, 2021, dtype=int))

# Chunking: do it once right before write (ok), but pick chunk sizes that match your access/write pattern
ds_all = ds_all.chunk({"init_time": 62, "year": 10, "lat": 21, "lon": 16})

In [8]:
ds_all

<xarray.Dataset>
Dimensions:         (lat: 21, lon: 16, year: 10, init_time: 62)
Coordinates:
  * lat             (lat) float64 55.13 56.07 57.02 57.96 ... 72.09 73.04 73.98
  * lon             (lon) float64 203.8 205.0 206.2 207.5 ... 220.0 221.2 222.5
  * year            (year) int64 0 1 2 3 4 5 6 7 8 9
  * init_time       (init_time) int64 1959 1960 1961 1962 ... 2018 2019 2020
Data variables:
    TREFHTMX_max    (init_time, year, lat, lon) float32 dask.array<chunksize=(62, 10, 21, 16), meta=np.ndarray>
    TREFHT_max      (init_time, year, lat, lon) float32 dask.array<chunksize=(62, 10, 21, 16), meta=np.ndarray>
    TREFHT_7d_max   (init_time, year, lat, lon) float32 dask.array<chunksize=(62, 10, 21, 16), meta=np.ndarray>
    TREFHT_30d_max  (init_time, year, lat, lon) float32 dask.array<chunksize=(62, 10, 21, 16), meta=np.ndarray>

In [9]:
save_name = dir_stn + "T2_max.zarr"
ds_all.to_zarr(save_name, mode="w")
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/Fairbanks/T2_max.zarr


In [10]:
ds_all = xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/METRICS/Fairbanks/T2_max.zarr')

In [13]:
# ds_all['TREFHTMX_max'].values